# **Step 1 — Housekeeping: Import Libraries and Set File Paths**

incomplete - waiting for values for BASE_URL; added assumed values. Might resubmit with the provided values.

In [1]:
import sqlite3
import csv
import urllib.request
import os

# GitHub raw URLs for the three CSV files
# Replace <username> and <repo> with the actual values provided by your instructor
BASE_URL = "https://raw.githubusercontent.com/melodylew/CIS3120/main/"

BOOK_URL   = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL   = BASE_URL + "Loan.csv"

# Local paths in the Colab /content directory

BOOK_PATH   = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH   = "/content/Loan.csv"

DB_PATH = "/content/library.db"

# **Step 2 — Download the CSV Files from GitHub**
Added assumed values. Might resubmit with the provided values.

In [2]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:
    urllib.request.urlretrieve(url, path)
    print(f"Downloaded: {path}")

Downloaded: /content/Book.csv
Downloaded: /content/Member.csv
Downloaded: /content/Loan.csv


# **Step 3 — Connect to the Database and Create Tables**

In [3]:
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')

# CREATE TABLE statements go here
conn.execute('''
    CREATE TABLE IF NOT EXISTS Book (
        callNo  TEXT    NOT NULL,
        title   TEXT    NOT NULL,
        author  TEXT    NOT NULL,
        PRIMARY KEY (callNo)
    );
''')

conn.execute('''
CREATE TABLE IF NOT EXISTS Member (
    id          INTEGER NOT NULL,
    firstname   TEXT    NOT NULL,
    lastName    TEXT    NOT NULL,
    PRIMARY KEY (id)
);
''')

conn.execute('''
CREATE TABLE IF NOT EXISTS Loan (
    callNo        TEXT    NOT NULL,
    id            INTEGER NOT NULL,
    dateBorrowed  TEXT    NOT NULL,
    dateReturned  TEXT,
    dateDue       TEXT    NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id)     REFERENCES Member(id)
);
''')

conn.commit()
print("Tables created.")

Tables created.


In [4]:
conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall()

[('Book',), ('Loan',), ('Member',)]

# **Step 4 — Load Book.csv into the Book Table**

In [5]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            'INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);',
            (row['callNo'], row['title'], row['author'])
        )

conn.commit()
print('Book rows loaded:', conn.execute('SELECT COUNT(*) FROM Book;').fetchone()[0])

Book rows loaded: 11


# **Step 5 — Load Member.csv into the Member Table**

In [6]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            'INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);',
            (int(row['id']), row['firstname'], row['lastName'])
        )

conn.commit()
print('Member rows loaded:', conn.execute('SELECT COUNT(*) FROM Member;').fetchone()[0])

Member rows loaded: 4


# **Step 6 — Load Loan.csv into the Loan Table**

In [7]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Convert empty dateReturned to None (→ NULL in SQLite)
        date_returned = row['dateReturned'] if row['dateReturned'].strip() else None

        conn.execute(
            '''INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
               VALUES (?, ?, ?, ?, ?);''',
            (row['callNo'], int(row['id']), row['dateBorrowed'], date_returned, row['dateDue'])
        )

conn.commit()
print('Loan rows loaded:', conn.execute('SELECT COUNT(*) FROM Loan;').fetchone()[0])

Loan rows loaded: 4


# **Query 1 — All Books**

Retrieve all columns from the Book table, ordered alphabetically by author last name (hint: SQLite has no last-name function — order by the author column as stored).

In [8]:
cursor = conn.execute('''
    SELECT *
    FROM Book
    ORDER BY author ASC;
''')

print("All rows in Book sorted by Author:")
for row in cursor:
    print(row)

All rows in Book sorted by Author:
('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


# **Query 2 — Books Not Yet Returned**

Retrieve the title of each book, and the first and last name of the member who borrowed it, for all loans where dateReturned is NULL (i.e., the book is still out).

This query requires a JOIN across all three tables.

In [9]:
cursor = conn.execute('''
    SELECT title,  firstname, lastname
    FROM Book
    JOIN Loan
    ON Book.callNo = Loan.callNo
    JOIN Member
    ON Loan.id = Member.id
    WHERE dateReturned IS NULL;
''')

print("Books Not Yet Returned:")
for row in cursor:
    print(row)

Books Not Yet Returned:
("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


# **Query 3 — Loan History for a Specific Book**

Retrieve the full loan history for the book with callNo R 141 E45 2006 — show the member's full name, dateBorrowed, dateDue, and dateReturned. Order by dateBorrowed ascending.

In [10]:
cursor = conn.execute('''
    SELECT firstname, lastname, dateBorrowed, dateDue, dateReturned
    FROM Book
    JOIN Loan
    ON Book.callNo = Loan.callNo
    JOIN Member
    ON Loan.id = Member.id
    WHERE Book.callNo = 'R 141 E45 2006'
    ORDER BY dateBorrowed ASC;
''')

print("Loan History for callNo R 141 E45 2006:")
for row in cursor:
    print(row)

Loan History for callNo R 141 E45 2006:
('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


# **Query 4 — Members Who Have Never Borrowed a Book**

Retrieve the id, firstname, and lastName of every member who does not appear in the Loan table. Use a LEFT JOIN or a sub-query with NOT IN.

In [11]:
cursor = conn.execute('''
    SELECT Member.id, firstname, lastname
    FROM Member
    LEFT JOIN Loan
    ON Member.id = Loan.id
    WHERE Loan.id IS NULL;
''')

print("Members Who Have Never Borrowed a Book:")
for row in cursor:
    print(row)

Members Who Have Never Borrowed a Book:
(4, 'John', 'Martin')


# **Query 5 — Count of Loans per Member**

Retrieve each member's full name and the total number of loans they have made (including completed ones). Include members with zero loans. Order by number of loans descending.

Hint: Use a LEFT JOIN from Member to Loan combined with COUNT() and GROUP BY.

In [12]:
cursor = conn.execute('''
    SELECT COUNT(Loan.id) AS num_loans, firstname, lastname
    FROM Member
    LEFT JOIN Loan
    ON Member.id = Loan.id
    GROUP BY Member.id
    ORDER BY num_loans DESC;
''')


print("Loans per Member:")
for row in cursor:
    print(row)

Loans per Member:
(2, 'David', 'Martin')
(1, 'John', 'Smith')
(1, 'Betty', 'Freeman')
(0, 'John', 'Martin')


# **Query 6 — Member with the Least Loans**

Finding the member with the least amount of loans identifies the members that the library needs to engage with more. The library can strategize how to redirect more promotional materials and reminders to engage and incentivize them to borrow a book.


In [13]:
cursor = conn.execute('''
    SELECT Member.id, Member.firstname, Member.lastname, COUNT(Loan.id) AS num_loans
    FROM Member
    LEFT JOIN Loan
    ON Member.id = Loan.id
    GROUP BY Member.id
    HAVING num_loans = (
        SELECT MIN(loan_count)
        FROM (
            SELECT COUNT(Loan.id) AS loan_count
            FROM Member
            LEFT JOIN Loan
            ON Member.id = Loan.id
            GROUP BY Member.id
        )
    )
    ORDER BY num_loans ASC;
''')

print("Member with the Least Loans:")
for row in cursor:
    print(row)

Member with the Least Loans:
(4, 'John', 'Martin', 0)


# **Closing the Connection**

In [14]:
conn.close()

# **Summary - Data Quality Observation**

One data quality observation I noticed is how Loan table is the bridge between the other tables because it shares callNo with the Book table and id with the Member table, allowing joins between the tables.

One data quality limitation is the formatting of the dates. I originally wanted to find the average loan duration, but the dates were not formatted in the standard ISO format, making it difficult to perform calculations.